# Four-Model Comparison — Snake-RepairLLaMA vs CodeLlama-base vs Kimi-Moonshot vs Gemini-2.5-flash

This notebook collates **all** evaluation signals across our four candidate models on **two benchmarks** (QuixBugs and BugsInPy):

| Signal      | What it measures                                                    | Cost      |
|-------------|---------------------------------------------------------------------|-----------|
| Exact match | Generated patch == gold patch byte-for-byte (after normalization)   | trivial   |
| AST match   | Generated patch has the same AST as gold (semantically identical)   | trivial   |
| Compile     | The function with the patch spliced in parses without SyntaxError   | trivial   |
| Plausibility| The patched program passes the project's own test suite             | expensive |

We re-use the project's `src.metrics.evaluate_file` helper for em / ast / compile, the `src.runners.quixbugs.run_plausibility` runner for QuixBugs plausibility (skipped if results already exist on disk), and the already-computed BugsInPy plausibility part files in `results/<model>/`.

Run cells top-to-bottom. The plausibility cells skip work that's already on disk.

## 0. Setup — paths, imports, repo root

In [24]:
from pathlib import Path
import os, sys, json, glob
from collections import defaultdict, Counter

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)

repo root: d:\snake-repairllama-baseline


In [26]:
# Per-model file inventory.
#  - 'gen'     : 1 row per bug, with a 'generations' list of 10 candidate patches.
#  - 'plaus'   : 1 row per (bug, gen_idx) with a test_pass field. May or may not
#                exist on disk yet — we'll fill it in below.
#
# For Kimi/Gemini the BugsInPy generation file is the *_aligned.jsonl variant
# (post-processed to fit the IR4/OR2 splice contract).
MODELS = {
    'Snakellama': {
        'quix_gen':   'results/snakellama/quixbugs_snakellama_generations.jsonl',
        'quix_plaus': 'results/snakellama/quixbugs_snakellama_plausibility.jsonl',
        'bugsinpy_gen': 'results/snakellama/bugsinpy_snakellama_generations.jsonl',
    },
    'CodeLlama-base': {
        'quix_gen':   'results/codellama-baseline/quixbugs_codellama_generations.jsonl',
        'quix_plaus': 'results/codellama-baseline/quixbugs_codellama_plausibility.jsonl',
        'bugsinpy_gen': 'results/codellama-baseline/bugsinpy_codellama_generations.jsonl',
    },
    'Kimi-Moonshot': {
        'quix_gen':   'results/kimi-moonshot/quixbugs_kimi_generations_aligned.jsonl',
        'quix_plaus': 'results/kimi-moonshot/quixbugs_kimi_plausibility.jsonl',
        'bugsinpy_gen': 'results/kimi-moonshot/bugsinpy_kimi_generations_aligned.jsonl',
    },
    'Gemini-2.5-flash': {
        'quix_gen':   'results/gemini-2.5-flash/quixbugs_gemini_generations_aligned.jsonl',
        'quix_plaus': 'results/gemini-2.5-flash/quixbugs_gemini_plausibility.jsonl',
        'bugsinpy_gen': 'results/gemini-2.5-flash/bugsinpy_gemini_generations_aligned.jsonl',
    },
}

QUIX_EVAL = 'data/quixbugs_eval.jsonl'
BIPY_EVAL = 'data/bugsinpy_eval_verified.jsonl'

# Sanity-check every generation file exists; flag missing plausibility files
# (those we may need to compute below).
for name, cfg in MODELS.items():
    for k in ('quix_gen', 'bugsinpy_gen'):
        assert Path(cfg[k]).exists(), f'{name}: missing {k} -> {cfg[k]}'
    print(f"  {name:<18s} quix-plaus-on-disk: {Path(cfg['quix_plaus']).exists()}")

  Snakellama         quix-plaus-on-disk: True
  CodeLlama-base     quix-plaus-on-disk: True
  Kimi-Moonshot      quix-plaus-on-disk: True
  Gemini-2.5-flash   quix-plaus-on-disk: True


## 1. QuixBugs plausibility — fill in any missing model

QuixBugs plausibility is cheap (~1-2 min per model) since the test harness is a single Python file per bug. Anything already on disk is **kept untouched**.

In [4]:
from src.runners.quixbugs import run_plausibility

for name, cfg in MODELS.items():
    out = Path(cfg['quix_plaus'])
    if out.exists():
        n = sum(1 for l in open(out, encoding='utf-8') if l.strip())
        print(f'  {name:<18s} SKIP — {out} already has {n} rows')
        continue
    print(f'  {name:<18s} RUN  -> {out}')
    out.parent.mkdir(parents=True, exist_ok=True)
    run_plausibility(
        eval_jsonl=QUIX_EVAL,
        inference_jsonl=cfg['quix_gen'],
        output_jsonl=str(out),
        start_bug=0,
        end_bug=40,
        timeout_sec=10,
        resume=True,
    )
print('done')

  Snakellama         SKIP — results\snakellama_on_quixbugs\quixbugs_snakellama_plausibility.jsonl already has 400 rows
  CodeLlama-base     SKIP — results\codellama-baseline\quixbugs_codellama_plausibility.jsonl already has 400 rows
  Kimi-Moonshot      SKIP — results\kimi-moonshot\quixbugs_kimi_plausibility.jsonl already has 400 rows
  Gemini-2.5-flash   SKIP — results\gemini-2.5-flash\quixbugs_gemini_plausibility.jsonl already has 400 rows
done


## 2. QuixBugs — em / ast / compile / plausibility for all 4 models

`evaluate_file` returns Top-1 / Top-3 / Top-10 counts. We pull plausibility from the per-model `*_plausibility.jsonl` so the same row contains all four signals.

In [5]:
from src.metrics import evaluate_file, print_report

quix_results = {}
for name, cfg in MODELS.items():
    plaus = cfg['quix_plaus'] if Path(cfg['quix_plaus']).exists() else None
    quix_results[name] = evaluate_file(
        inference_jsonl=cfg['quix_gen'],
        eval_jsonl=QUIX_EVAL,
        plausibility_jsonl=plaus,
    )
    print_report(f'QuixBugs — {name}', quix_results[name])


  QuixBugs — Snakellama
  Total bugs: 40    Bugs with plausibility data: 40
  Top-1  Exact     :   19 / 40 ( 47.5%)
  Top-1  AST       :   21 / 40 ( 52.5%)
  Top-1  Compile   :   39 / 40 ( 97.5%)
  Top-1  Plausible :   23 / 40 ( 57.5%) [tests passed]

  Top-3  Exact     :   28 / 40 ( 70.0%)
  Top-3  AST       :   30 / 40 ( 75.0%)
  Top-3  Compile   :   39 / 40 ( 97.5%)
  Top-3  Buried    :   28 / 40 ( 70.0%)
  Top-3  Plausible :   34 / 40 ( 85.0%)

  Top-10 Exact     :   32 / 40 ( 80.0%)
  Top-10 AST       :   33 / 40 ( 82.5%)
  Top-10 Compile   :   40 / 40 (100.0%)
  Top-10 Buried    :   32 / 40 ( 80.0%)
  Top-10 Plausible :   36 / 40 ( 90.0%)

  QuixBugs — CodeLlama-base
  Total bugs: 40    Bugs with plausibility data: 40
  Top-1  Exact     :    0 / 40 (  0.0%)
  Top-1  AST       :    6 / 40 ( 15.0%)
  Top-1  Compile   :   26 / 40 ( 65.0%)
  Top-1  Plausible :   12 / 40 ( 30.0%) [tests passed]

  Top-3  Exact     :    1 / 40 (  2.5%)
  Top-3  AST       :   12 / 40 ( 30.0%)
  Top-3  

In [6]:
# Side-by-side QuixBugs summary (Top-10 — most informative for pass@10 framing).
import pandas as pd

def row_for(res, n):
    c = res['counts']
    n_p = res.get('n_with_plausibility', 0) or n
    return {
        'Top-1 EM':      f"{c['top1_exact']}/{n} ({100*c['top1_exact']/n:.1f}%)",
        'Top-1 AST':     f"{c['top1_ast']}/{n} ({100*c['top1_ast']/n:.1f}%)",
        'Top-1 Compile': f"{c['top1_compile']}/{n} ({100*c['top1_compile']/n:.1f}%)",
        'Top-1 Plaus':   f"{c['top1_plausible']}/{n_p} ({100*c['top1_plausible']/n_p:.1f}%)" if n_p else 'n/a',
        'Top-10 EM':     f"{c['top10_exact']}/{n} ({100*c['top10_exact']/n:.1f}%)",
        'Top-10 AST':    f"{c['top10_ast']}/{n} ({100*c['top10_ast']/n:.1f}%)",
        'Top-10 Compile':f"{c['top10_compile']}/{n} ({100*c['top10_compile']/n:.1f}%)",
        'Top-10 Plaus':  f"{c['top10_plausible']}/{n_p} ({100*c['top10_plausible']/n_p:.1f}%)" if n_p else 'n/a',
    }

rows = {name: row_for(res, res['n']) for name, res in quix_results.items()}
df_quix = pd.DataFrame(rows).T
df_quix.index.name = 'Model'
df_quix

,Top-1 EM,Top-1 AST,Top-1 Compile,Top-1 Plaus,Top-10 EM,Top-10 AST,Top-10 Compile,Top-10 Plaus
Model,,,,,,,,
Snakellama,19/40 (47.5%),21/40 (52.5%),39/40 (97.5%),23/40 (57.5%),32/40 (80.0%),33/40 (82.5%),40/40 (100.0%),36/40 (90.0%)
CodeLlama-base,0/40 (0.0%),6/40 (15.0%),26/40 (65.0%),12/40 (30.0%),4/40 (10.0%),27/40 (67.5%),39/40 (97.5%),31/40 (77.5%)
Kimi-Moonshot,22/40 (55.0%),22/40 (55.0%),36/40 (90.0%),28/40 (70.0%),32/40 (80.0%),33/40 (82.5%),39/40 (97.5%),36/40 (90.0%)
Gemini-2.5-flash,30/40 (75.0%),31/40 (77.5%),34/40 (85.0%),34/40 (85.0%),30/40 (75.0%),31/40 (77.5%),34/40 (85.0%),34/40 (85.0%)


### 2a. QuixBugs per-bug analysis

For each of the 40 bugs, did the model produce **any** patch in its 10 candidates that:

- compiles (Top-10 Compile)?
- AST-matches gold?
- passes tests (plausibility)?

Cells show `c` (compile only), `a` (also AST-matches gold), `P` (also passes tests), `.` (none). Sorting by Snakellama success keeps the easy bugs at the top.

In [7]:
def per_bug_marks(res):
    out = {}
    for rec in res['per_bug']:
        bid = rec['bug_id']
        any_compile = any(s['compile'] for s in rec['scores'])
        any_ast     = any(s['ast']     for s in rec['scores'])
        any_plaus   = any(p is True    for p in rec['plausibility'])
        if any_plaus:    mark = 'P'
        elif any_ast:    mark = 'a'
        elif any_compile:mark = 'c'
        else:            mark = '.'
        out[bid] = mark
    return out

marks = {name: per_bug_marks(res) for name, res in quix_results.items()}
all_bugs = sorted({b for m in marks.values() for b in m.keys()})
rank = {'P': 0, 'a': 1, 'c': 2, '.': 3}
all_bugs.sort(key=lambda b: (rank.get(marks['Snakellama'].get(b, '.'), 3), b))

rows = []
for b in all_bugs:
    rows.append({'bug_id': b, **{m: marks[m].get(b, '.') for m in MODELS}})
df_quix_per_bug = pd.DataFrame(rows).set_index('bug_id')
df_quix_per_bug

,Snakellama,CodeLlama-base,Kimi-Moonshot,Gemini-2.5-flash
bug_id,,,,
quixbugs/bitcount,P,P,P,P
quixbugs/breadth_first_search,P,P,P,P
quixbugs/bucketsort,P,P,P,P
quixbugs/depth_first_search,P,P,P,.
quixbugs/detect_cycle,P,P,P,P
quixbugs/find_first_in_sorted,P,P,P,P
quixbugs/find_in_sorted,P,P,P,P
quixbugs/flatten,P,P,P,P
quixbugs/gcd,P,P,P,P


In [8]:
# Plausibility agreement: for each bug, how many of the 4 models pass tests?
agreement = Counter()
any_pass_set = defaultdict(set)
for b in all_bugs:
    passers = [m for m in MODELS if marks[m].get(b) == 'P']
    agreement[len(passers)] += 1
    for m in passers:
        any_pass_set[m].add(b)

print('Bugs solved by N models (plausibility):')
for k in sorted(agreement):
    print(f'  {k} models: {agreement[k]:>2d} bugs')

# Bugs ONLY Snakellama solves
for m in MODELS:
    others = set().union(*[any_pass_set[o] for o in MODELS if o != m])
    only_m = any_pass_set[m] - others
    if only_m:
        print(f'\nQuix bugs ONLY {m} solves ({len(only_m)}): {sorted(only_m)}')

Bugs solved by N models (plausibility):
  0 models:  1 bugs
  1 models:  1 bugs
  2 models:  3 bugs
  3 models: 10 bugs
  4 models: 25 bugs

Quix bugs ONLY Snakellama solves (1): ['quixbugs/powerset']


## 3. BugsInPy — em / ast / compile for all 4 models

BugsInPy plausibility is computed in a separate docker pipeline (see `scripts/bugsinpy_run_eval.py` + `scripts/run_bugsinpy_4workers.py`) and lives under `results/<model>/bugsinpy_*_plausibility.jsonl`. The static metrics below are independent of plausibility — they tell us how often the model produces *anything* gold-equivalent or even syntactically valid for these real-world bugs.

In [9]:
bipy_results = {}
for name, cfg in MODELS.items():
    bipy_results[name] = evaluate_file(
        inference_jsonl=cfg['bugsinpy_gen'],
        eval_jsonl=BIPY_EVAL,
        plausibility_jsonl=None,
    )
    print_report(f'BugsInPy — {name}', bipy_results[name])

<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.



  BugsInPy — Snakellama
  Total bugs: 161    Bugs with plausibility data: 0
  Top-1  Exact     :   11 / 161 (  6.8%)
  Top-1  AST       :   12 / 161 (  7.5%)
  Top-1  Compile   :  155 / 161 ( 96.3%)

  Top-3  Exact     :   14 / 161 (  8.7%)
  Top-3  AST       :   15 / 161 (  9.3%)
  Top-3  Compile   :  157 / 161 ( 97.5%)
  Top-3  Buried    :   17 / 161 ( 10.6%)

  Top-10 Exact     :   23 / 161 ( 14.3%)
  Top-10 AST       :   25 / 161 ( 15.5%)
  Top-10 Compile   :  157 / 161 ( 97.5%)
  Top-10 Buried    :   28 / 161 ( 17.4%)


<unknown>:5: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.



  BugsInPy — CodeLlama-base
  Total bugs: 161    Bugs with plausibility data: 0
  Top-1  Exact     :    0 / 161 (  0.0%)
  Top-1  AST       :    1 / 161 (  0.6%)
  Top-1  Compile   :   85 / 161 ( 52.8%)

  Top-3  Exact     :    0 / 161 (  0.0%)
  Top-3  AST       :    4 / 161 (  2.5%)
  Top-3  Compile   :  130 / 161 ( 80.7%)
  Top-3  Buried    :    8 / 161 (  5.0%)

  Top-10 Exact     :    0 / 161 (  0.0%)
  Top-10 AST       :    9 / 161 (  5.6%)
  Top-10 Compile   :  147 / 161 ( 91.3%)
  Top-10 Buried    :   15 / 161 (  9.3%)


<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.



  BugsInPy — Kimi-Moonshot
  Total bugs: 161    Bugs with plausibility data: 0
  Top-1  Exact     :    6 / 161 (  3.7%)
  Top-1  AST       :    6 / 161 (  3.7%)
  Top-1  Compile   :  129 / 161 ( 80.1%)

  Top-3  Exact     :   13 / 161 (  8.1%)
  Top-3  AST       :   13 / 161 (  8.1%)
  Top-3  Compile   :  145 / 161 ( 90.1%)
  Top-3  Buried    :   19 / 161 ( 11.8%)

  Top-10 Exact     :   17 / 161 ( 10.6%)
  Top-10 AST       :   17 / 161 ( 10.6%)
  Top-10 Compile   :  153 / 161 ( 95.0%)
  Top-10 Buried    :   24 / 161 ( 14.9%)

  BugsInPy — Gemini-2.5-flash
  Total bugs: 161    Bugs with plausibility data: 0
  Top-1  Exact     :   13 / 161 (  8.1%)
  Top-1  AST       :   13 / 161 (  8.1%)
  Top-1  Compile   :  107 / 161 ( 66.5%)

  Top-3  Exact     :   18 / 161 ( 11.2%)
  Top-3  AST       :   18 / 161 ( 11.2%)
  Top-3  Compile   :  120 / 161 ( 74.5%)
  Top-3  Buried    :   20 / 161 ( 12.4%)

  Top-10 Exact     :   20 / 161 ( 12.4%)
  Top-10 AST       :   21 / 161 ( 13.0%)
  Top-10 Comp

In [10]:
rows = {name: row_for(res, res['n']) for name, res in bipy_results.items()}
df_bipy = pd.DataFrame(rows).T
df_bipy.index.name = 'Model'
# Drop the Plausibility columns here — we'll show the proper docker-pipeline
# plausibility numbers in section 4 below.
df_bipy = df_bipy.drop(columns=[c for c in df_bipy.columns if 'Plaus' in c])
df_bipy

,Top-1 EM,Top-1 AST,Top-1 Compile,Top-10 EM,Top-10 AST,Top-10 Compile
Model,,,,,,
Snakellama,11/161 (6.8%),12/161 (7.5%),155/161 (96.3%),23/161 (14.3%),25/161 (15.5%),157/161 (97.5%)
CodeLlama-base,0/161 (0.0%),1/161 (0.6%),85/161 (52.8%),0/161 (0.0%),9/161 (5.6%),147/161 (91.3%)
Kimi-Moonshot,6/161 (3.7%),6/161 (3.7%),129/161 (80.1%),17/161 (10.6%),17/161 (10.6%),153/161 (95.0%)
Gemini-2.5-flash,13/161 (8.1%),13/161 (8.1%),107/161 (66.5%),20/161 (12.4%),21/161 (13.0%),125/161 (77.6%)


### 3a. BugsInPy per-bug snapshot (Top-10)

Same legend as QuixBugs but plausibility is omitted here — see Section 4 for docker-runner plausibility. `a` = any of 10 gens AST-matches gold; `c` = any compiles; `.` = none.

In [11]:
def per_bug_marks_static(res):
    out = {}
    for rec in res['per_bug']:
        bid = rec['bug_id']
        any_compile = any(s['compile'] for s in rec['scores'])
        any_ast     = any(s['ast']     for s in rec['scores'])
        if any_ast:      mark = 'a'
        elif any_compile:mark = 'c'
        else:            mark = '.'
        out[bid] = mark
    return out

bmarks = {name: per_bug_marks_static(res) for name, res in bipy_results.items()}
all_bipy_bugs = sorted({b for m in bmarks.values() for b in m.keys()})
rank2 = {'a': 0, 'c': 1, '.': 2}
all_bipy_bugs.sort(key=lambda b: (rank2.get(bmarks['Snakellama'].get(b, '.'), 2), b))
rows = [{'bug_id': b, **{m: bmarks[m].get(b, '.') for m in MODELS}} for b in all_bipy_bugs]
df_bipy_per_bug = pd.DataFrame(rows).set_index('bug_id')
df_bipy_per_bug.head(40)

,Snakellama,CodeLlama-base,Kimi-Moonshot,Gemini-2.5-flash
bug_id,,,,
ansible/15,a,c,a,c
cookiecutter/1,a,c,c,a
cookiecutter/3,a,c,.,a
fastapi/12,a,.,.,.
fastapi/5,a,.,a,a
keras/38,a,a,a,a
luigi/13,a,a,c,c
luigi/28,a,c,c,c
matplotlib/24,a,c,.,c


## 4. BugsInPy plausibility (docker-pipeline results)

These come from `scripts/bugsinpy_run_eval.py` running each model's generated patches through the project-specific test suite under docker. Each model is restricted to the **71 bugs reproducible from Snakellama** (i.e. bugs where Snakellama's gold patch made the project's tests pass cleanly — our anchor set for apples-to-apples comparison).

We re-load directly from the per-worker part files and compute pass@10 (≥1 of 10 gens passes the test suite).

In [12]:
def load_bipy_plaus(gen_glob, gold_glob):
    gold_pass, gen_pass, gen_total = set(), defaultdict(int), defaultdict(int)
    statuses = defaultdict(lambda: defaultdict(int))
    for f in sorted(glob.glob(gold_glob)):
        with open(f, encoding='utf-8') as fh:
            for l in fh:
                if l.strip():
                    r = json.loads(l)
                    if r['gold_test_status'] == 'pass':
                        gold_pass.add(r['bug_id'])
    for f in sorted(glob.glob(gen_glob)):
        with open(f, encoding='utf-8') as fh:
            for l in fh:
                if l.strip():
                    r = json.loads(l)
                    bid = r['bug_id']
                    st  = r['test_status'].replace('_via_dedup', '')
                    gen_total[bid] += 1
                    statuses[bid][st] += 1
                    if st == 'pass':
                        gen_pass[bid] += 1
    return {'gold_pass': gold_pass, 'gen_pass': gen_pass,
            'gen_total': gen_total, 'statuses': statuses}

BIPY_PLAUS = {
    'Snakellama': load_bipy_plaus(
        'results/snakellama_on_reproducible_bugsinpy/gen_results.jsonl',
        'results/snakellama_on_reproducible_bugsinpy/gold.jsonl'),
    'CodeLlama-base': load_bipy_plaus(
        'results/codellama-baseline/bugsinpy_codellama_gen_part*.jsonl',
        'results/codellama-baseline/bugsinpy_codellama_gold_part*.jsonl'),
    'Kimi-Moonshot': load_bipy_plaus(
        'results/kimi-moonshot/bugsinpy_kimi_gen_part*.jsonl',
        'results/kimi-moonshot/bugsinpy_kimi_gold_part*.jsonl'),
    'Gemini-2.5-flash': load_bipy_plaus(
        'results/gemini-2.5-flash/bugsinpy_gemini_gen_part*.jsonl',
        'results/gemini-2.5-flash/bugsinpy_gemini_gold_part*.jsonl'),
}

REPRO = set(open('results/reproducible_bug_ids.txt').read().split())
print(f'Reproducible-bug anchor set: {len(REPRO)} bugs')
for name, p in BIPY_PLAUS.items():
    print(f'  {name:<18s} gold-pass: {len(p["gold_pass"])}/71, evaluated: {len(p["gen_total"])}')

Reproducible-bug anchor set: 71 bugs
  Snakellama         gold-pass: 71/71, evaluated: 71
  CodeLlama-base     gold-pass: 71/71, evaluated: 71
  Kimi-Moonshot      gold-pass: 71/71, evaluated: 71
  Gemini-2.5-flash   gold-pass: 71/71, evaluated: 71


In [13]:
# Two views — common 4-way intersection (apples-to-apples for the per-gen
# distribution) and the snakellama-anchor view (counts a bug as 0/10 for any
# model that failed to even reproduce gold).
common = REPRO.copy()
for p in BIPY_PLAUS.values():
    common &= set(p['gen_total'].keys())

def pass_at_10(p, bugs):
    return sum(1 for b in bugs if p['gen_pass'].get(b, 0) > 0)

def per_gen_dist(p, bugs):
    tot = sum(sum(p['statuses'][b].values()) for b in bugs)
    pa  = sum(p['statuses'][b].get('pass', 0) for b in bugs)
    fa  = sum(p['statuses'][b].get('fail', 0) for b in bugs)
    er  = tot - pa - fa
    return tot, pa, fa, er

def fmt(num, den): return f'{num}/{den} ({100*num/max(den,1):.1f}%)'

rows_common, rows_anchor = {}, {}
n_c, n_a = len(common), len(REPRO)
for name, p in BIPY_PLAUS.items():
    pa10_c = pass_at_10(p, common)
    pa10_a = sum(1 for b in REPRO if b in p['gold_pass'] and p['gen_pass'].get(b,0)>0)
    tot, pa, fa, er = per_gen_dist(p, common)
    rows_common[name] = {
        'pass@10': fmt(pa10_c, n_c),
        'per-gen pass':  f'{100*pa/tot:.1f}%',
        'per-gen fail':  f'{100*fa/tot:.1f}%',
        'per-gen error': f'{100*er/tot:.1f}%',
    }
    rows_anchor[name] = {'pass@10 (71-anchor)': fmt(pa10_a, n_a)}

df_common = pd.DataFrame(rows_common).T
df_common.index.name = f'Model (common {n_c} bugs)'
df_anchor = pd.DataFrame(rows_anchor).T
df_anchor.index.name = f'Model (full {n_a}-bug anchor)'
print('=== BugsInPy plausibility — apples-to-apples (common evaluated bugs) ===')
display(df_common)
print('\n=== BugsInPy plausibility — Snakellama-anchor view (missing = 0/10) ===')
display(df_anchor)

=== BugsInPy plausibility — apples-to-apples (common evaluated bugs) ===


,pass@10,per-gen pass,per-gen fail,per-gen error
Model (common 71 bugs),,,,
Snakellama,36/71 (50.7%),14.5%,80.3%,5.2%
CodeLlama-base,24/71 (33.8%),8.9%,50.8%,40.3%
Kimi-Moonshot,24/71 (33.8%),12.3%,65.8%,22.0%
Gemini-2.5-flash,29/71 (40.8%),24.5%,38.5%,37.0%



=== BugsInPy plausibility — Snakellama-anchor view (missing = 0/10) ===


,pass@10 (71-anchor)
Model (full 71-bug anchor),
Snakellama,36/71 (50.7%)
CodeLlama-base,24/71 (33.8%)
Kimi-Moonshot,24/71 (33.8%)
Gemini-2.5-flash,29/71 (40.8%)


### 4a. BugsInPy plausibility — per-bug breakdown

In [14]:
# For each bug in the 71-anchor set, show per-model #pass / 10 (or '-' if the
# model couldn't reproduce gold for that bug — i.e. the env didn't compile).
def cell(p, b):
    if b not in p['gen_total']:
        return '-'
    if b not in p['gold_pass']:
        return 'g!'  # gen evaluated but gold did not pass
    return f"{p['gen_pass'].get(b,0)}/10"

rows = []
for b in sorted(REPRO, key=lambda x: (x.split('/')[0], int(x.split('/')[1]))):
    rows.append({'bug_id': b, **{name: cell(p, b) for name, p in BIPY_PLAUS.items()}})
df_bipy_plaus_per_bug = pd.DataFrame(rows).set_index('bug_id')

# Sort: bugs Snakellama solves first, then by total #models that solve
def sort_key(b):
    snake = BIPY_PLAUS['Snakellama']['gen_pass'].get(b, 0)
    total = sum(BIPY_PLAUS[m]['gen_pass'].get(b, 0) > 0 for m in MODELS)
    return (-int(snake>0), -total, b)
df_bipy_plaus_per_bug = df_bipy_plaus_per_bug.loc[sorted(df_bipy_plaus_per_bug.index, key=sort_key)]
df_bipy_plaus_per_bug.head(71)

,Snakellama,CodeLlama-base,Kimi-Moonshot,Gemini-2.5-flash
bug_id,,,,
ansible/15,7/10,1/10,9/10,2/10
ansible/3,1/10,2/10,8/10,10/10
black/21,7/10,4/10,6/10,10/10
keras/9,1/10,1/10,1/10,10/10
matplotlib/7,3/10,6/10,1/10,9/10
...,...,...,...,...
thefuck/4,0/10,0/10,0/10,0/10
youtube-dl/1,0/10,0/10,0/10,0/10
youtube-dl/27,0/10,0/10,0/10,0/10


In [15]:
# Solver agreement on BugsInPy plausibility.
any_pass = {name: {b for b in REPRO if BIPY_PLAUS[name]['gen_pass'].get(b,0) > 0}
            for name in MODELS}

agreement = Counter()
for b in REPRO:
    agreement[sum(b in any_pass[m] for m in MODELS)] += 1
print('Bugs solved (≥1/10 plausible) by N of the 4 models:')
for k in sorted(agreement, reverse=True):
    print(f'  {k} models: {agreement[k]:>2d} / {len(REPRO)} bugs')

for m in MODELS:
    others = set().union(*[any_pass[o] for o in MODELS if o != m])
    only_m = any_pass[m] - others
    if only_m:
        print(f'\nBugsInPy bugs ONLY {m} solves ({len(only_m)}): {sorted(only_m)}')

Bugs solved (≥1/10 plausible) by N of the 4 models:
  4 models: 13 / 71 bugs
  3 models:  5 / 71 bugs
  2 models: 18 / 71 bugs
  1 models: 10 / 71 bugs
  0 models: 25 / 71 bugs

BugsInPy bugs ONLY Snakellama solves (6): ['black/10', 'httpie/4', 'thefuck/19', 'thefuck/9', 'youtube-dl/25', 'youtube-dl/43']

BugsInPy bugs ONLY Kimi-Moonshot solves (1): ['youtube-dl/41']

BugsInPy bugs ONLY Gemini-2.5-flash solves (3): ['scrapy/38', 'youtube-dl/10', 'youtube-dl/20']


## 5. Headlines

Pulled programmatically from the tables above so they stay in sync if you re-run on updated data.

In [16]:
print('QuixBugs (40 bugs):')
for name, res in quix_results.items():
    n = res['n']; c = res['counts']; n_p = res.get('n_with_plausibility', 0) or n
    print(f"  {name:<18s} Top-10: EM={c['top10_exact']}/{n}  AST={c['top10_ast']}/{n}  "
          f"Compile={c['top10_compile']}/{n}  Plausible={c['top10_plausible']}/{n_p}")
print()
print('BugsInPy static (161 bugs):')
for name, res in bipy_results.items():
    n = res['n']; c = res['counts']
    print(f"  {name:<18s} Top-10: EM={c['top10_exact']}/{n}  AST={c['top10_ast']}/{n}  "
          f"Compile={c['top10_compile']}/{n}")
print()
print(f'BugsInPy plausibility (Snakellama-anchor, {len(REPRO)} bugs):')
for name, p in BIPY_PLAUS.items():
    pa = sum(1 for b in REPRO if b in p['gold_pass'] and p['gen_pass'].get(b,0)>0)
    print(f"  {name:<18s} pass@10 = {pa}/{len(REPRO)} = {100*pa/len(REPRO):.1f}%")

QuixBugs (40 bugs):
  Snakellama         Top-10: EM=32/40  AST=33/40  Compile=40/40  Plausible=36/40
  CodeLlama-base     Top-10: EM=4/40  AST=27/40  Compile=39/40  Plausible=31/40
  Kimi-Moonshot      Top-10: EM=32/40  AST=33/40  Compile=39/40  Plausible=36/40
  Gemini-2.5-flash   Top-10: EM=30/40  AST=31/40  Compile=34/40  Plausible=34/40

BugsInPy static (161 bugs):
  Snakellama         Top-10: EM=23/161  AST=25/161  Compile=157/161
  CodeLlama-base     Top-10: EM=0/161  AST=9/161  Compile=147/161
  Kimi-Moonshot      Top-10: EM=17/161  AST=17/161  Compile=153/161
  Gemini-2.5-flash   Top-10: EM=20/161  AST=21/161  Compile=125/161

BugsInPy plausibility (Snakellama-anchor, 71 bugs):
  Snakellama         pass@10 = 36/71 = 50.7%
  CodeLlama-base     pass@10 = 24/71 = 33.8%
  Kimi-Moonshot      pass@10 = 24/71 = 33.8%
  Gemini-2.5-flash   pass@10 = 29/71 = 40.8%


## 6. Combined: solved-by-EM ∪ solved-by-plausibility (Top-10)

A bug is **solved** if **either** signal fires in the model's 10 candidates:

* **EM hit** — some generation, after normalization, equals gold byte-for-byte. EM is a **gold-side** signal: it works on every bug in the eval set, including bugs whose project env we couldn't reproduce.
* **Plaus hit** — some generation passes the project's own test suite under docker. This requires the env to be reproducible (only 71/161 BugsInPy bugs).

The union is the cleanest "did the model actually fix this bug" answer because:

* Bugs with broken envs (PyPI rot, OpenSSL incompat, etc.) still count if the model wrote the gold patch.
* Bugs whose gold patch is overly strict (e.g. style differences acceptable to the test suite) still count if a *different* generation passes the tests.

Per-bug rule: `solved := (Top-10 EM hit) OR (any of 10 gens has test_pass=True)`.

### 6a. QuixBugs union (40 bugs)

For QuixBugs every bug is reproducible, so the union usually equals plausibility — but EM occasionally lifts a bug whose patch is gold-equivalent but flaky in the test harness.

In [17]:
# Build per-bug EM-hit and Plaus-hit sets per model.
def union_marks_quix():
    rows_summary = []
    rows_per_bug = {}
    for name, res in quix_results.items():
        em_set, plaus_set = set(), set()
        for rec in res['per_bug']:
            bid = rec['bug_id']
            is_em = any(s['exact'] for s in rec['scores'])
            is_p  = any(p is True for p in rec['plausibility'])
            if is_em: em_set.add(bid)
            if is_p:  plaus_set.add(bid)
            rows_per_bug.setdefault(bid, {})[name] = (
                'EP' if (is_em and is_p)
                else 'P' if is_p
                else 'E' if is_em
                else '.'
            )
        union = em_set | plaus_set
        n = res['n']
        rows_summary.append({
            'Model': name,
            'EM@10':       f'{len(em_set)}/{n} ({100*len(em_set)/n:.1f}%)',
            'Plaus@10':    f'{len(plaus_set)}/{n} ({100*len(plaus_set)/n:.1f}%)',
            'Union @10':   f'{len(union)}/{n} ({100*len(union)/n:.1f}%)',
            'EM-only lift':len(em_set - plaus_set),
        })
    return rows_summary, rows_per_bug

quix_summary, quix_per_bug = union_marks_quix()
df_quix_union = pd.DataFrame(quix_summary).set_index('Model')
print('=== QuixBugs Union (EM ∪ Plausible) @10 ===')
display(df_quix_union)

=== QuixBugs Union (EM ∪ Plausible) @10 ===


,EM@10,Plaus@10,Union @10,EM-only lift
Model,,,,
Snakellama,32/40 (80.0%),36/40 (90.0%),36/40 (90.0%),0
CodeLlama-base,4/40 (10.0%),31/40 (77.5%),31/40 (77.5%),0
Kimi-Moonshot,32/40 (80.0%),36/40 (90.0%),36/40 (90.0%),0
Gemini-2.5-flash,30/40 (75.0%),34/40 (85.0%),34/40 (85.0%),0


In [18]:
# Per-bug for QuixBugs: 'EP' = both, 'E' = EM only, 'P' = plaus only, '.' = neither.
all_bugs_q = sorted(quix_per_bug.keys())
order = {'EP': 0, 'P': 1, 'E': 2, '.': 3}
all_bugs_q.sort(key=lambda b: (order.get(quix_per_bug[b].get('Snakellama', '.'), 3), b))
rows = [{'bug_id': b, **{m: quix_per_bug[b].get(m, '.') for m in MODELS}} for b in all_bugs_q]
df_quix_union_per_bug = pd.DataFrame(rows).set_index('bug_id')
df_quix_union_per_bug

,Snakellama,CodeLlama-base,Kimi-Moonshot,Gemini-2.5-flash
bug_id,,,,
quixbugs/bitcount,EP,P,EP,P
quixbugs/breadth_first_search,EP,P,EP,EP
quixbugs/bucketsort,EP,P,EP,EP
quixbugs/depth_first_search,EP,P,EP,.
quixbugs/detect_cycle,EP,P,EP,EP
quixbugs/find_first_in_sorted,EP,EP,EP,EP
quixbugs/find_in_sorted,EP,P,EP,EP
quixbugs/flatten,EP,P,EP,EP
quixbugs/gcd,EP,EP,EP,EP


### 6b. BugsInPy union (full 161-bug eval set)

This is the headline metric for BugsInPy: **a bug is solved if any of 10 gens matches gold OR any of 10 gens passes the project tests**. We compute over the full 161-bug eval set so EM-hits on non-reproducible bugs (the ~90 bugs we couldn't get docker envs for) still count.

Plausibility is sourced from the docker-pipeline part files, keyed by `(bug_id, gen_idx)`. EM is the static metric from `evaluate_file`.

In [19]:
# Build (bug_id -> set of plausible gen_idx) for each model from docker part files.
import glob as _glob
def load_plaus_index(gen_glob):
    idx = defaultdict(set)
    for f in sorted(_glob.glob(gen_glob)):
        with open(f, encoding='utf-8') as fh:
            for l in fh:
                if not l.strip(): continue
                r = json.loads(l)
                st = r['test_status'].replace('_via_dedup', '')
                if st == 'pass':
                    idx[r['bug_id']].add(r.get('gen_idx', -1))
    return idx

BIPY_PLAUS_GENIDX = {
    'Snakellama':       load_plaus_index('results/snakellama_on_reproducible_bugsinpy/gen_results.jsonl'),
    'CodeLlama-base':   load_plaus_index('results/codellama-baseline/bugsinpy_codellama_gen_part*.jsonl'),
    'Kimi-Moonshot':    load_plaus_index('results/kimi-moonshot/bugsinpy_kimi_gen_part*.jsonl'),
    'Gemini-2.5-flash': load_plaus_index('results/gemini-2.5-flash/bugsinpy_gemini_gen_part*.jsonl'),
}
for name, d in BIPY_PLAUS_GENIDX.items():
    n_bugs_with_plaus = sum(1 for v in d.values() if v)
    print(f'  {name:<18s} bugs with at least 1 plausible gen: {n_bugs_with_plaus}')

  Snakellama         bugs with at least 1 plausible gen: 36
  CodeLlama-base     bugs with at least 1 plausible gen: 24
  Kimi-Moonshot      bugs with at least 1 plausible gen: 24
  Gemini-2.5-flash   bugs with at least 1 plausible gen: 29


In [20]:
# Compute Union @10 for BugsInPy on the full eval set (n=161).
bipy_union_summary, bipy_union_per_bug = [], {}
for name, res in bipy_results.items():
    em_set, plaus_set = set(), set()
    plaus_idx = BIPY_PLAUS_GENIDX[name]
    for rec in res['per_bug']:
        bid = rec['bug_id']
        is_em = any(s['exact'] for s in rec['scores'])
        is_p  = bool(plaus_idx.get(bid))
        if is_em: em_set.add(bid)
        if is_p:  plaus_set.add(bid)
        bipy_union_per_bug.setdefault(bid, {})[name] = (
            'EP' if (is_em and is_p)
            else 'P' if is_p
            else 'E' if is_em
            else '.'
        )
    union = em_set | plaus_set
    n = res['n']
    bipy_union_summary.append({
        'Model': name,
        'EM@10':         f'{len(em_set)}/{n} ({100*len(em_set)/n:.1f}%)',
        'Plaus@10':      f'{len(plaus_set)}/{n} ({100*len(plaus_set)/n:.1f}%)',
        'Union @10':     f'{len(union)}/{n} ({100*len(union)/n:.1f}%)',
        'EM-only lift':  len(em_set - plaus_set),
    })

df_bipy_union = pd.DataFrame(bipy_union_summary).set_index('Model')
print('=== BugsInPy Union (EM ∪ Plausible) @10 — full 161-bug eval set ===')
display(df_bipy_union)
print()
print('  EM-only lift = bugs the model fixes via gold-equivalent patch but plausibility')
print('  test failed/missing (broken env, flaky tests, gen_idx mismatch).')

=== BugsInPy Union (EM ∪ Plausible) @10 — full 161-bug eval set ===


,EM@10,Plaus@10,Union @10,EM-only lift
Model,,,,
Snakellama,23/161 (14.3%),36/161 (22.4%),53/161 (32.9%),17
CodeLlama-base,0/161 (0.0%),24/161 (14.9%),24/161 (14.9%),0
Kimi-Moonshot,17/161 (10.6%),24/161 (14.9%),35/161 (21.7%),11
Gemini-2.5-flash,20/161 (12.4%),29/161 (18.0%),43/161 (26.7%),14



  EM-only lift = bugs the model fixes via gold-equivalent patch but plausibility
  test failed/missing (broken env, flaky tests, gen_idx mismatch).


In [21]:
# Per-bug breakdown for BugsInPy union, sorted so Snakellama-solved bugs come first.
all_bugs_b = sorted(bipy_union_per_bug.keys(), key=lambda x: (x.split('/')[0], int(x.split('/')[1])))
order = {'EP': 0, 'P': 1, 'E': 2, '.': 3}
all_bugs_b.sort(key=lambda b: (order.get(bipy_union_per_bug[b].get('Snakellama', '.'), 3), b))
rows = [{'bug_id': b, **{m: bipy_union_per_bug[b].get(m, '.') for m in MODELS}} for b in all_bugs_b]
df_bipy_union_per_bug = pd.DataFrame(rows).set_index('bug_id')
df_bipy_union_per_bug

,Snakellama,CodeLlama-base,Kimi-Moonshot,Gemini-2.5-flash
bug_id,,,,
ansible/15,EP,P,EP,P
thefuck/19,EP,.,.,.
thefuck/2,EP,.,P,EP
tornado/14,EP,P,EP,EP
youtube-dl/12,EP,P,EP,EP
...,...,...,...,...
youtube-dl/20,.,.,.,P
youtube-dl/27,.,.,.,.
youtube-dl/3,.,.,.,.


In [22]:
# Solver-agreement on BugsInPy union.
solved_by = {name: {b for b, marks in bipy_union_per_bug.items() if marks.get(name, '.') in ('E','P','EP')}
             for name in MODELS}
agree = Counter()
for b in bipy_union_per_bug:
    agree[sum(b in solved_by[m] for m in MODELS)] += 1
print('BugsInPy bugs solved (EM or plausible) by N of the 4 models:')
for k in sorted(agree, reverse=True):
    print(f'  {k} models: {agree[k]:>3d} / {sum(agree.values())} bugs')
print()
for m in MODELS:
    others = set().union(*[solved_by[o] for o in MODELS if o != m])
    only_m = solved_by[m] - others
    if only_m:
        suffix = ' ...' if len(only_m) > 20 else ''
        print(f'BugsInPy bugs ONLY {m} solves ({len(only_m)}): {sorted(only_m)[:20]}{suffix}')

BugsInPy bugs solved (EM or plausible) by N of the 4 models:
  4 models:  13 / 161 bugs
  3 models:  10 / 161 bugs
  2 models:  26 / 161 bugs
  1 models:  21 / 161 bugs
  0 models:  91 / 161 bugs

BugsInPy bugs ONLY Snakellama solves (14): ['black/10', 'fastapi/12', 'httpie/4', 'luigi/13', 'luigi/28', 'matplotlib/24', 'matplotlib/26', 'pandas/28', 'pandas/4', 'pandas/74', 'thefuck/19', 'thefuck/9', 'youtube-dl/25', 'youtube-dl/43']
BugsInPy bugs ONLY Kimi-Moonshot solves (2): ['pandas/1', 'youtube-dl/41']
BugsInPy bugs ONLY Gemini-2.5-flash solves (5): ['luigi/24', 'scrapy/32', 'scrapy/38', 'youtube-dl/10', 'youtube-dl/20']


### 6c. Final headline — union score across both benchmarks

In [23]:
print('=== UNION (EM ∪ Plausible) @10 ===')
print()
print(f'{"Model":<20s} {"QuixBugs (40)":<22s} {"BugsInPy (161)":<22s}')
for name in MODELS:
    qrow = next(r for r in quix_summary if r['Model'] == name)
    brow = next(r for r in bipy_union_summary if r['Model'] == name)
    print(f'  {name:<18s} {qrow["Union @10"]:<22s} {brow["Union @10"]:<22s}')

=== UNION (EM ∪ Plausible) @10 ===

Model                QuixBugs (40)          BugsInPy (161)        
  Snakellama         36/40 (90.0%)          53/161 (32.9%)        
  CodeLlama-base     31/40 (77.5%)          24/161 (14.9%)        
  Kimi-Moonshot      36/40 (90.0%)          35/161 (21.7%)        
  Gemini-2.5-flash   34/40 (85.0%)          43/161 (26.7%)        
